## 06 -- LSTM Regressor

Bidirectional LSTM trained on 24-hour sliding windows of PM2.5 and weather
features. Uses `StandardScaler` fit on training data only, `EarlyStopping`
with `patience=5`, and temporal train/test split (test = 2024-Q4).

In [1]:
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report
warnings.filterwarnings('ignore')

In [2]:
PROC = pathlib.Path('../data/processed')
df = pd.read_csv(PROC / 'features_engineered.csv', parse_dates=['datetime'])

# LSTM uses raw (non-engineered) features -- the sequence captures temporal context
LSTM_FEATURE_COLS = [
    'pm25', 'o3', 'no2',
    'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
    'wind_direction_10m', 'precipitation', 'surface_pressure',
    'hour', 'day_of_week', 'month', 'is_weekend',
]
TARGET_COL = 'pm25_next24h'
LOOKBACK   = 24  # hours of history per sample

model_df = df.dropna(subset=[TARGET_COL]).copy()

# Forward-fill any gaps in LSTM input features per city
for city in sorted(model_df['city'].unique()):
    mask = model_df['city'] == city
    model_df.loc[mask, LSTM_FEATURE_COLS] = (
        model_df.loc[mask, LSTM_FEATURE_COLS].ffill().bfill()
    )

print(f'Rows: {len(model_df):,}   Cities: {sorted(model_df["city"].unique())}')
model_df[['datetime','city'] + LSTM_FEATURE_COLS].head(3)

Rows: 81,028   Cities: ['Birmingham A4540 Roadside', 'Edinburgh St Leonards', 'Leeds Centre', 'London Marylebone Road', 'Manchester Piccadilly']


,datetime,city,pm25,o3,no2,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,surface_pressure,hour,day_of_week,month,is_weekend
0,2023-01-01 01:00:00,Birmingham A4540 Roadside,7.123,61.33451,14.17608,8.5,89,29.2,218,0.0,981.7,1,6,1,1
1,2023-01-01 02:00:00,Birmingham A4540 Roadside,4.057,65.79158,10.04362,8.0,87,27.3,222,0.0,982.7,2,6,1,1
2,2023-01-01 03:00:00,Birmingham A4540 Roadside,4.363,66.40692,11.48356,7.3,88,25.5,220,0.0,983.5,3,6,1,1


In [3]:
SPLIT_DATE = '2024-10-01'

train_df = model_df[model_df['datetime'] < SPLIT_DATE].copy()
test_df  = model_df[model_df['datetime'] >= SPLIT_DATE].copy()

print(f'Train: {len(train_df):,} rows    Test: {len(test_df):,} rows')

# Fit scaler on training data only
scaler_X = StandardScaler()
scaler_y = StandardScaler()

train_df[LSTM_FEATURE_COLS]  = scaler_X.fit_transform(train_df[LSTM_FEATURE_COLS])
train_df[[TARGET_COL]]       = scaler_y.fit_transform(train_df[[TARGET_COL]])
test_df[LSTM_FEATURE_COLS]   = scaler_X.transform(test_df[LSTM_FEATURE_COLS])
test_df[[TARGET_COL]]        = scaler_y.transform(test_df[[TARGET_COL]])

print('Features scaled with StandardScaler (fit on train).')

Train: 70,224 rows    Test: 10,804 rows
Features scaled with StandardScaler (fit on train).


In [4]:
def make_sequences(city_df, lookback):
    city_df = city_df.sort_values('datetime').reset_index(drop=True)
    feat = city_df[LSTM_FEATURE_COLS].values.astype('float32')
    tgt  = city_df[TARGET_COL].values.astype('float32')
    X, y = [], []
    for i in range(lookback, len(city_df)):
        X.append(feat[i - lookback:i])
        y.append(tgt[i])
    return np.array(X, dtype='float32'), np.array(y, dtype='float32')

X_tr_parts, y_tr_parts = [], []
X_te_parts, y_te_parts = [], []

for city in sorted(model_df['city'].unique()):
    X_tr, y_tr = make_sequences(train_df[train_df['city'] == city], LOOKBACK)
    X_te, y_te = make_sequences(test_df[test_df['city']  == city], LOOKBACK)
    X_tr_parts.append(X_tr);  y_tr_parts.append(y_tr)
    X_te_parts.append(X_te);  y_te_parts.append(y_te)

X_train_lstm = np.concatenate(X_tr_parts)
y_train_lstm = np.concatenate(y_tr_parts)
X_test_lstm  = np.concatenate(X_te_parts)
y_test_lstm  = np.concatenate(y_te_parts)

print(f'Train sequences: X={X_train_lstm.shape}  y={y_train_lstm.shape}')
print(f'Test  sequences: X={X_test_lstm.shape}   y={y_test_lstm.shape}')

Train sequences: X=(70104, 24, 13)  y=(70104,)
Test  sequences: X=(10684, 24, 13)   y=(10684,)


In [5]:
import keras
from keras import layers

n_features = len(LSTM_FEATURE_COLS)

model = keras.Sequential([
    layers.Input(shape=(LOOKBACK, n_features)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
], name='lstm_pm25')

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "lstm_pm25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        19,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,929 (128.63 KB)

 Trainable params: 32,929 (128.63 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
from keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

history = model.fit(
    X_train_lstm, y_train_lstm,
    epochs=30,
    batch_size=512,
    validation_split=0.1,
    callbacks=[es],
    verbose=1,
)

# Training loss curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history['loss'],     label='train loss')
ax.plot(history.history['val_loss'], label='val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('LSTM Training Loss')
ax.legend()
plt.tight_layout()
pathlib.Path('../models').mkdir(exist_ok=True)
plt.savefig('../models/lstm_training_loss.png', dpi=120)
plt.show()
print('Training complete.')

Epoch 1/30


  1/124 ━━━━━━━━━━━━━━━━━━━━ 4:33 2s/step - loss: 0.9012 - mae: 0.7133

  2/124 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 1.0204 - mae: 0.7268

  3/124 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 0.9849 - mae: 0.7240

  4/124 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - loss: 0.9538 - mae: 0.7163

...

122/124 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.3775 - mae: 0.4545

124/124 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - loss: 0.3777 - mae: 0.4544 - val_loss: 0.4555 - val_mae: 0.4852


Epoch 8: early stopping


Restoring model weights from the end of the best epoch: 3.


Training complete.


In [7]:
y_pred_scaled = model.predict(X_test_lstm).ravel()

# Inverse-transform to original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_test = scaler_y.inverse_transform(y_test_lstm.reshape(-1, 1)).ravel()
y_pred = np.clip(y_pred, 0, None)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
print(f'Test RMSE : {rmse:.3f} ug/m3')
print(f'Test MAE  : {mae:.3f} ug/m3')

def daqi_band(x):
    if x < 12:   return 'Low'
    elif x < 24: return 'Moderate'
    elif x < 48: return 'High'
    else:        return 'Very High'

bands = ['Low', 'Moderate', 'High', 'Very High']
y_true_band = [daqi_band(v) for v in y_test]
y_pred_band = [daqi_band(v) for v in y_pred]
print()
print('DAQI band classification:')
print(classification_report(y_true_band, y_pred_band, labels=bands, zero_division=0))

  1/334 ━━━━━━━━━━━━━━━━━━━━ 57s 171ms/step

 18/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step   

 35/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

 52/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

 70/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

...

305/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

320/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

334/334 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


Test RMSE : 6.405 ug/m3
Test MAE  : 4.407 ug/m3

DAQI band classification:
              precision    recall  f1-score   support

         Low       0.79      0.97      0.87      8443
    Moderate       0.24      0.04      0.07      1851
        High       0.00      0.00      0.00       387
   Very High       0.00      0.00      0.00         3

    accuracy                           0.78     10684
   macro avg       0.26      0.25      0.24     10684
weighted avg       0.67      0.78      0.70     10684



In [8]:
import joblib
MODEL_DIR = pathlib.Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
model.save(str(MODEL_DIR / 'lstm.keras'))
joblib.dump({'scaler_X': scaler_X, 'scaler_y': scaler_y, 'feature_cols': LSTM_FEATURE_COLS},
            MODEL_DIR / 'lstm_scalers.pkl')
print('Model saved to models/lstm.keras')
print('Scalers saved to models/lstm_scalers.pkl')

Model saved to models/lstm.keras
Scalers saved to models/lstm_scalers.pkl
